# Data Preparation Pipeline
## Pneumonia Detection Project — Group 4

End-to-end data preparation for the pneumonia CNN: from raw images sitting in S3 through preprocessed PNGs registered in a SageMaker Feature Store, ending with a simple benchmark model trained on summary statistics.

**Pipeline stages in this notebook**
1. **Environment setup** — install AWS SDK pins and confirm the data lake is populated (assumes `data-setup.ipynb` has been run).
2. **File-format inventory** — quick S3 scan to confirm the mix of DICOM and JPEG matches expectations.
3. **Metadata table (Athena)** — build a per-image manifest from S3 + the RSNA labels CSV, register it as an Athena external table for SQL exploration.
4. **Exploratory data analysis** — class balance, sample images, size distribution, pixel intensity.
5. **Image preprocessing** — DICOM/JPEG → grayscale → CLAHE → resize → PNG; recompute pixel stats per image and ingest into SageMaker Feature Store.
6. **Canonical train/val/test/production split** — stratified 40/10/10/40 on the preprocessed manifest.
7. **Benchmark model** — a sanity-check SKLearn classifier on pixel_mean / pixel_std to establish a floor before the CNN.

**Side effects:** writes to `s3://pneumonia-data-set-group-4/` (raw scan), `s3://<default-bucket>/` (preprocessed PNGs, manifest, Feature Store, benchmark artifacts). The notebook uses an `IS_DATA_OWNER` kill-switch (default `False`) to gate writes that affect the shared bucket.


## Step 1 · Environment Setup

Pin `sagemaker` to the 2.x line and install the AWS data-access libraries the notebook needs (`pyathena` for SQL-over-S3, `awswrangler` for pandas-friendly AWS helpers). The `boto3` floor guards against a stale base image that ships an old AWS SDK.

In [1]:
%pip uninstall sagemaker -y
%pip install "sagemaker>=2.0,<3.0" -q

In [2]:
!pip install "sagemaker<3" pyathena awswrangler  --quiet
!pip install 'boto3>1.17.21' -q

## Step 2 · Verify the S3 Data Lake

Confirm that `data-setup.ipynb` has populated the raw-images prefix and that the expected `train/val/test × NORMAL/PNEUMONIA` folder structure exists. A zero count here means the upstream notebook hasn't been run (or hit an error).

In [3]:
import boto3

s3 = boto3.client("s3")
bucket = "pneumonia-data-set-group-4"

# Check what top-level folders exist
response = s3.list_objects_v2(Bucket=bucket, Prefix="raw-images/", Delimiter="/")
print("Top folders:")
for prefix in response.get("CommonPrefixes", []):
    print(f"  {prefix['Prefix']}")

# Count files in each folder
paginator = s3.get_paginator("list_objects_v2")
for split in ["train", "test", "val"]:
    for label in ["NORMAL", "PNEUMONIA"]:
        prefix = f"raw-images/{split}/{label}/"
        count = 0
        for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
            count += len(page.get("Contents", []))
        if count > 0:
            print(f"  {split}/{label}: {count} files")

### File-format inventory

Two source datasets means two formats — Kermany ships JPEG, RSNA ships DICOM. Count each so the downstream preprocessing pipeline knows what it has to handle. (`dcm_keys` is also captured for the spot-check below.)

In [9]:
import boto3

s3 = boto3.client("s3")
bucket = "pneumonia-data-set-group-4"
paginator = s3.get_paginator("list_objects_v2")

dcm_count = 0
jpeg_count = 0
other_count = 0
dcm_keys = []
for page in paginator.paginate(Bucket=bucket, Prefix="raw-images/"):
    for obj in page.get("Contents", []):
        key = obj["Key"]
        if key.endswith(".dcm"):
            dcm_count += 1
            dcm_keys.append(key)
        elif key.endswith((".jpeg", ".jpg", ".png")):
            jpeg_count += 1
        else:
            other_count += 1

print(f"DICOM (.dcm):  {dcm_count}")
print(f"JPEG (.jpeg):  {jpeg_count}")
print(f"Other:         {other_count}")
print(f"Total:         {dcm_count + jpeg_count + other_count}")

Spot-check the first DICOM key to confirm the path shape (split/label/file.dcm).

In [10]:
dcm_keys[0]

## Step 3 · Build the Metadata Table in Athena

Amazon Athena is serverless SQL over S3. Instead of looping over S3 keys every time we want a class count or a per-source breakdown, we register a single metadata CSV — one row per image — as an Athena external table and query it with SQL.

**What this section does:**
1. Connect PyAthena to a query-results staging directory.
2. Create the `pneumonia_db` database in the AWS Glue catalog.
3. Build the metadata DataFrame: walk S3 for images, derive labels (RSNA from the labels CSV, Kermany from the folder name), drop unlabeled rows.
4. Upload the CSV (gated behind `IS_DATA_OWNER`) and register the Athena table.
5. Run verification queries.


### 3.1 Connect to Athena and create the database

In [5]:
import boto3
import sagemaker
import pandas as pd
from pyathena import connect

In [6]:
#Setup Connection



sess = sagemaker.Session()
default_bucket = sess.default_bucket()
region = boto3.Session().region_name
bucket = "pneumonia-data-set-group-4"

# Athena staging directory (uses YOUR default bucket for query results)
s3_staging_dir = f"s3://{default_bucket}/athena/staging"

# Connect to Athena
conn = connect(region_name=region, s3_staging_dir=s3_staging_dir)

print(f"Region: {region}")
print(f"Data bucket: {bucket}")
print(f"Staging dir: {s3_staging_dir}")

In [7]:
#Create the database

database_name = "pneumonia_db"

statement = f"CREATE DATABASE IF NOT EXISTS {database_name}"
print(statement)
pd.read_sql(statement, conn)
print("✅ Database created!")

In [8]:
df_databases = pd.read_sql("SHOW DATABASES", conn)
print(df_databases)

if database_name in df_databases.values:
    print(f"\n✅ Database '{database_name}' exists!")
else:
    print(f"\n❌ Database '{database_name}' not found")

### 3.2 Build the per-image metadata DataFrame

Combine the authoritative RSNA labels (from the CSVs uploaded to S3 by `data-setup.ipynb` §5.4) with a sweep of the `raw-images/` prefix. The result is one row per image with `image_id`, `s3_key`, `source`, `label`, `label_int`, and `file_size`.

In [ ]:
# Load the RSNA labels from S3 (uploaded by data-setup.ipynb Section 5.4).
# Two CSVs: stage_2_train_labels.csv covers the training set; stage_2_sample_submission.csv
# covers the held-out test set. Concatenating them gives us a single patientId -> Target map.
rsna_metadata_prefix = 'raw-metadata/rsna'

import awswrangler as wr
rsna_df_train = wr.s3.read_csv(f's3://{bucket}/{rsna_metadata_prefix}/stage_2_train_labels.csv')
rsna_df_sub   = wr.s3.read_csv(f's3://{bucket}/{rsna_metadata_prefix}/stage_2_sample_submission.csv')
rsna_df = pd.concat([rsna_df_train, rsna_df_sub], ignore_index=True)

# Multiple bounding-box rows per patient collapse to a single label here — Target is 0/1.
# Drop duplicates so the map lookup is unambiguous.
rsna_label_map = dict(zip(rsna_df.patientId, rsna_df.Target))
print(f'RSNA label map: {len(rsna_label_map)} patient IDs')

In [ ]:
# Build the unified metadata DataFrame by walking every image under raw-images/.
# RSNA labels come from the map we just built; Kermany labels come from the S3 path
# (the dataset has no separate labels CSV — folder name IS the ground truth).
s3 = boto3.client('s3')
paginator = s3.get_paginator('list_objects_v2')
rows = []

for page in paginator.paginate(Bucket=bucket, Prefix='raw-images/'):
    for obj in page.get('Contents', []):
        key = obj['Key']
        file_name = key.split('/')[-1]
        ext = file_name.rsplit('.', 1)[-1].lower()
        if ext not in ('jpeg', 'jpg', 'dcm'):
            continue  # skip non-image files (e.g., .ipynb_checkpoints)

        image_id = file_name.rsplit('.', 1)[0]
        if ext == 'dcm':
            source = 'rsna'
            label_int = rsna_label_map.get(image_id)
        else:
            source = 'chest_xray'
            label_int = 0 if 'NORMAL' in key else 1

        rows.append({
            'image_id':  image_id,
            's3_key':    key,
            'file_name': file_name,
            'file_type': 'dcm' if ext == 'dcm' else 'jpeg',
            'source':    source,
            'label_int': label_int,
            'file_size': obj['Size'],
        })

df_metadata = pd.DataFrame(rows)
print(f'Total images scanned: {len(df_metadata)}')
df_metadata.head()

In [ ]:
# Drop rows where the RSNA label lookup returned NaN (rare — would mean the patient
# ID isn't in either CSV; shouldn't happen for either Kermany or RSNA train+sub).
before = len(df_metadata)
df_metadata = df_metadata.dropna(subset=['label_int']).copy()
df_metadata['label_int'] = df_metadata['label_int'].astype(int)
print(f'Dropped {before - len(df_metadata)} rows with missing labels; {len(df_metadata)} remain.')

In [ ]:
# Convert numeric label back to the canonical string for downstream readability.
label_int_to_str = {0: 'NORMAL', 1: 'PNEUMONIA'}
df_metadata['label'] = df_metadata['label_int'].map(label_int_to_str)

df_metadata['is_preprocessed'] = False
df_metadata['file_size'] = df_metadata['file_size'].astype(int)

df_metadata.label.value_counts()

### 3.3 Publish the metadata CSV to S3

In [ ]:
# IS_DATA_OWNER kill-switch keeps this notebook re-runnable safely for graders /
# teammates — the read-only path verifies the metadata can be built, while the data
# owner flips this to True to actually publish to S3.
IS_DATA_OWNER = False

if IS_DATA_OWNER:
    csv_path = 'image_metadata.csv'
    df_metadata.to_csv(csv_path, index=False)
    s3.upload_file(csv_path, bucket, f'pneumonia-project/metadata/{csv_path}')
    print(f'Uploaded s3://{bucket}/pneumonia-project/metadata/{csv_path}')
else:
    print('IS_DATA_OWNER=False — skipping upload. Flip to True to publish to S3.')

### 3.4 Register the table in Athena

In [ ]:
# Register the metadata CSV as an external Athena table so we can query class
# balance and source breakdown with SQL instead of scanning S3 each time.
# `split` is intentionally NOT a column here — the canonical train/val/test/production
# split happens after preprocessing in Step 7.
statement = f"""
CREATE EXTERNAL TABLE IF NOT EXISTS {database_name}.image_metadata (
    image_id          STRING,
    s3_key            STRING,
    file_name         STRING,
    file_type         STRING,
    source            STRING,
    label_int         TINYINT,
    label             STRING,
    file_size         BIGINT,
    is_preprocessed   BOOLEAN
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ','
LOCATION 's3://{bucket}/pneumonia-project/metadata/'
TBLPROPERTIES ('skip.header.line.count'='1')
"""
pd.read_sql(statement, conn)
print(f'Created table {database_name}.image_metadata')

In [10]:
df_tables = pd.read_sql(f"SHOW TABLES IN {database_name}", conn)
print(df_tables)

if "image_metadata" in df_tables.values:
    print("\n✅ Table 'image_metadata' exists!")
else:
    print("\n❌ Table not found")

### 3.5 Verify with SQL queries

In [ ]:
# Verification queries: confirm the table is queryable and the label/source mix
# matches what we built in pandas.
df_count = pd.read_sql(f'SELECT COUNT(*) AS total_images FROM {database_name}.image_metadata', conn)
print('Total images:')
print(df_count)

df_labels = pd.read_sql(
    f'SELECT label, COUNT(*) AS count FROM {database_name}.image_metadata GROUP BY label',
    conn,
)
print('\nClass distribution:')
print(df_labels)

df_breakdown = pd.read_sql(
    f'''SELECT source, label, COUNT(*) AS count
        FROM {database_name}.image_metadata
        GROUP BY source, label
        ORDER BY source, label''',
    conn,
)
print('\nBreakdown by source and label:')
print(df_breakdown)

## Step 4 · Exploratory Data Analysis

Sanity-check the combined dataset against the assumptions a CNN would make: class balance (will we need class weights?), image size variation (will resize introduce artifacts?), pixel intensity spread (does CLAHE help?), and whether the two sources look visually consistent.

Findings here are summarized in the final cell of this section and used to justify the preprocessing choices in Step 5.


In [5]:
!pip install pydicom opencv-python-headless matplotlib seaborn --quiet

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 1.5.0 requires nvidia-ml-py3<8.0,>=7.352.0, which is not installed.
autogluon-timeseries 1.5.0 requires chronos-forecasting<2.4,>=2.2.2, which is not installed.
autogluon-timeseries 1.5.0 requires einops<1,>=0.7, which is not installed.
autogluon-timeseries 1.5.0 requires peft<0.18,>=0.13.0, which is not installed.
skops 0.14.0 requires prettytable>=3.9, which is not installed.
amazon-sagemaker-sql-magic 0.1.4 requires numpy<2, but you have numpy 2.4.6 which is incompatible.
autogluon-common 1.5.0 requires numpy<2.4.0,>=1.25.0, but you have numpy 2.4.6 which is incompatible.
autogluon-common 1.5.0 requires pyarrow<21.0.0,>=7.0.0, but you have pyarrow 21.0.0 which is incompatible.
autogluon-core 1.5.0 requires numpy<2.4.0,>=1.25.0, but you have numpy 2.4.6 which is incompatible.
autogluon-featu

In [6]:
import boto3
import numpy as np
import cv2
import pydicom
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import io
import random

s3 = boto3.client("s3")
bucket = "pneumonia-data-set-group-4"

print("✅ Setup complete")

✅ Setup complete


In [ ]:
# this code is repeated from up further up? can this be deleted? -sc
# Build metadata from S3 
paginator = s3.get_paginator("list_objects_v2")
rows = []

for split in ["train", "test", "val"]:
    for label in ["NORMAL", "PNEUMONIA"]:
        prefix = f"raw-images/{split}/{label}/"
        for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
            for obj in page.get("Contents", []):
                key = obj["Key"]
                if ".ipynb_checkpoint" in key:
                    continue  # skip junk files
                file_name = key.split("/")[-1]
                file_type = "dcm" if file_name.endswith(".dcm") else "jpeg"
                source = "rsna" if file_type == "dcm" else "chest_xray"

                rows.append({
                    "image_id": file_name.replace(".dcm", "").replace(".jpeg", "").replace(".jpg", ""),
                    "s3_key": key,
                    "file_name": file_name,
                    "split": split,
                    "label": label,
                    "file_type": file_type,
                    "source": source,
                    "file_size": obj["Size"]
                })

df = pd.DataFrame(rows)
print(f"Total images: {len(df)}")
df.head()

### 4.1 Dataset overview (counts by label, split, source)

In [ ]:
#Dataset overview

print("=" * 50)
print("DATASET OVERVIEW")
print("=" * 50)

print(f"\nTotal images: {len(df)}")

print(f"\nBy label:")
print(df["label"].value_counts())

print(f"\nBy split:")
print(df["split"].value_counts())

print(f"\nBy source:")
print(df["source"].value_counts())

normal = len(df[df["label"] == "NORMAL"])
pneumonia = len(df[df["label"] == "PNEUMONIA"])
print(f"\nClass ratio - Normal:Pneumonia = {normal/pneumonia:.2f}:1")

### 4.2 Class distribution charts

In [ ]:
#Class distribution charts

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Overall class balance
counts = df["label"].value_counts()
axes[0].bar(counts.index, counts.values, color=["steelblue", "salmon"])
axes[0].set_title("Overall Class Distribution")
axes[0].set_ylabel("Count")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 100, str(v), ha="center", fontweight="bold")

# By split
split_counts = df.groupby(["split", "label"]).size().unstack(fill_value=0)
split_counts.plot(kind="bar", ax=axes[1], color=["steelblue", "salmon"])
axes[1].set_title("Class Distribution by Split")
axes[1].set_ylabel("Count")
axes[1].tick_params(axis="x", rotation=0)
axes[1].legend(title="Label")

# By source
source_counts = df.groupby(["source", "label"]).size().unstack(fill_value=0)
source_counts.plot(kind="bar", ax=axes[2], color=["steelblue", "salmon"])
axes[2].set_title("Class Distribution by Source")
axes[2].set_ylabel("Count")
axes[2].tick_params(axis="x", rotation=0)
axes[2].legend(title="Label")

plt.tight_layout()
plt.show()

### 4.3 Helper: load images from S3

Single function that handles both formats: DICOM gets `pydicom` + min-max normalization; JPEG gets `cv2.imdecode` in grayscale mode.

In [ ]:
#helper function to load images from S3

def load_image_from_s3(s3_key):
    """Load an image from S3 - handles both JPEG and DICOM formats"""
    response = s3.get_object(Bucket=bucket, Key=s3_key)
    img_bytes = response["Body"].read()

    if s3_key.endswith(".dcm"):
        # DICOM format (RSNA dataset)
        ds = pydicom.dcmread(io.BytesIO(img_bytes))
        img = ds.pixel_array
        img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    else:
        # JPEG format (Chest X-Ray dataset)
        img_array = np.frombuffer(img_bytes, np.uint8)
        img = cv2.imdecode(img_array, cv2.IMREAD_GRAYSCALE)

    return img

# Test it
test_key = df.iloc[0]["s3_key"]
test_img = load_image_from_s3(test_key)
print(f"✅ Loaded image — shape: {test_img.shape}, dtype: {test_img.dtype}")

### 4.4 Sample X-rays (Normal vs Pneumonia)

In [ ]:
#Visualize sample images:

fig, axes = plt.subplots(2, 5, figsize=(20, 8))

normal_samples = df[df["label"] == "NORMAL"].sample(5, random_state=42)
pneumonia_samples = df[df["label"] == "PNEUMONIA"].sample(5, random_state=42)

# Normal images (top row)
for i, (_, row) in enumerate(normal_samples.iterrows()):
    img = load_image_from_s3(row["s3_key"])
    axes[0][i].imshow(img, cmap="gray")
    axes[0][i].set_title(f"NORMAL\n{row['source']}\n{img.shape}", fontsize=9)
    axes[0][i].axis("off")

# Pneumonia images (bottom row)
for i, (_, row) in enumerate(pneumonia_samples.iterrows()):
    img = load_image_from_s3(row["s3_key"])
    axes[1][i].imshow(img, cmap="gray")
    axes[1][i].set_title(f"PNEUMONIA\n{row['source']}\n{img.shape}", fontsize=9)
    axes[1][i].axis("off")

plt.suptitle("Sample Images: Normal vs Pneumonia", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### 4.5 Image size distribution

Image dimensions vary considerably — we sample 500 images to characterize the spread before choosing a resize target.

In [ ]:
#Image size distribution
# Sample 500 images to check sizes

sample_df = df.sample(min(500, len(df)), random_state=42)

heights, widths, sources, labels = [], [], [], []

print("Analyzing image sizes ")
for _, row in sample_df.iterrows():
    try:
        img = load_image_from_s3(row["s3_key"])
        heights.append(img.shape[0])
        widths.append(img.shape[1])
        sources.append(row["source"])
        labels.append(row["label"])
    except:
        continue

size_df = pd.DataFrame({"height": heights, "width": widths, "source": sources, "label": labels})

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(data=size_df, x="height", hue="source", ax=axes[0], bins=30)
axes[0].set_title("Image Height Distribution")

sns.histplot(data=size_df, x="width", hue="source", ax=axes[1], bins=30)
axes[1].set_title("Image Width Distribution")

sns.scatterplot(data=size_df, x="width", y="height", hue="source", alpha=0.5, ax=axes[2])
axes[2].set_title("Height vs Width")

plt.tight_layout()
plt.show()

print("\nImage size stats:")
print(size_df.groupby("source")[["height", "width"]].describe())

### 4.6 Pixel intensity by class

Pneumonia X-rays should trend toward mid-range intensities (fluid in lungs scatters X-rays into greys). If the distributions look identical, our preprocessing should preserve detail aggressively (this is why CLAHE).

In [ ]:
#Pixel intensity analysis

sample_normal = df[df["label"] == "NORMAL"].sample(50, random_state=42)
sample_pneumonia = df[df["label"] == "PNEUMONIA"].sample(50, random_state=42)

normal_means, pneumonia_means = [], []
normal_stds, pneumonia_stds = [], []

print("Analyzing pixel intensities...")

for _, row in sample_normal.iterrows():
    try:
        img = load_image_from_s3(row["s3_key"])
        normal_means.append(img.mean())
        normal_stds.append(img.std())
    except:
        continue

for _, row in sample_pneumonia.iterrows():
    try:
        img = load_image_from_s3(row["s3_key"])
        pneumonia_means.append(img.mean())
        pneumonia_stds.append(img.std())
    except:
        continue

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(normal_means, bins=20, alpha=0.6, label="Normal", color="steelblue")
axes[0].hist(pneumonia_means, bins=20, alpha=0.6, label="Pneumonia", color="salmon")
axes[0].set_title("Mean Pixel Intensity Distribution")
axes[0].set_xlabel("Mean Pixel Value")
axes[0].legend()

axes[1].hist(normal_stds, bins=20, alpha=0.6, label="Normal", color="steelblue")
axes[1].hist(pneumonia_stds, bins=20, alpha=0.6, label="Pneumonia", color="salmon")
axes[1].set_title("Pixel Intensity Std Distribution")
axes[1].set_xlabel("Std Pixel Value")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Normal    — Mean: {np.mean(normal_means):.1f}, Std: {np.mean(normal_stds):.1f}")
print(f"Pneumonia — Mean: {np.mean(pneumonia_means):.1f}, Std: {np.mean(pneumonia_stds):.1f}")

### 4.7 EDA Summary

In [ ]:
print("=" * 50)
print("EDA SUMMARY")
print("=" * 50)
print(f"""
Total Images:     {len(df)}
  - Normal:       {len(df[df['label']=='NORMAL'])} ({len(df[df['label']=='NORMAL'])/len(df)*100:.1f}%)
  - Pneumonia:    {len(df[df['label']=='PNEUMONIA'])} ({len(df[df['label']=='PNEUMONIA'])/len(df)*100:.1f}%)

Sources:
  - Chest X-Ray:  {len(df[df['source']=='chest_xray'])} JPEG images
  - RSNA:         {len(df[df['source']=='rsna'])} DICOM images

Splits:
  - Train:        {len(df[df['split']=='train'])}
  - Test:         {len(df[df['split']=='test'])}
  - Val:          {len(df[df['split']=='val'])}

Key Findings:
  1. Class imbalance: {len(df[df['label']=='NORMAL'])/len(df[df['label']=='PNEUMONIA']):.2f}:1 ratio (Normal:Pneumonia)
  2. Two file formats need unified preprocessing (DICOM + JPEG)
  3. Image sizes vary — need resizing to 512x512 (per config.py)
""")

## Step 5 · Image Preprocessing + SageMaker Feature Store

Transform raw X-rays into model-ready PNGs and store a queryable **training manifest** in SageMaker Feature Store.

**Preprocessing pipeline (applied per image):**
1. Read from S3 (handles both DICOM and JPEG).
2. Min-max normalize to uint8 (collapses 12/16-bit DICOM to a consistent 8-bit range).
3. CLAHE contrast enhancement (`clipLimit=2.0`, `tileGridSize=(8,8)`) — surfaces subtle opacities without amplifying noise.
4. Resize to 512×512 (uniform CNN input).
5. Encode as PNG (lossless) and write to S3.

**Why Feature Store instead of plain CSV?** We're using a CNN, so the model learns its own features from raw pixels. The "features" we register are really a *manifest*: image_id → preprocessed S3 key, label, split, and pixel mean/std. This gives the training pipeline a queryable lookup table instead of forcing it to scan S3 folders and parse paths.


In [ ]:
!pip install pydicom opencv-python-headless --quiet

### 5.1 Setup: clients, role, default bucket

In [7]:
# Setup

import boto3
import sagemaker
import pandas as pd
import numpy as np
import cv2
import pydicom
import io
import time
import tempfile
import os
from datetime import datetime
from sagemaker.feature_store.feature_group import FeatureGroup

sess = sagemaker.Session()
role = sagemaker.get_execution_role()
region = boto3.Session().region_name
default_bucket = sess.default_bucket()
s3 = boto3.client("s3")
bucket = "pneumonia-data-set-group-4"

print(f"Role: {role}")
print(f"Region: {region}")
print(f"Default bucket: {default_bucket}")

Role: arn:aws:iam::072753725776:role/LabRole
Region: us-east-1
Default bucket: sagemaker-us-east-1-072753725776


### 5.2 Helpers: load + preprocess

Same I/O helper from §4.3 plus the canonical 4-step preprocessing pipeline. Tested on a single image before running on the full set.

In [8]:
# helper functions to load images from S3 and preprocess them for CNN training

IMG_SIZE = (512, 512)

def load_image_from_s3(s3_key):
    """Load image from S3 - handles both JPEG and DICOM"""
    response = s3.get_object(Bucket=bucket, Key=s3_key)
    img_bytes = response["Body"].read()

    if s3_key.endswith(".dcm"):
        ds = pydicom.dcmread(io.BytesIO(img_bytes))
        img = ds.pixel_array
    else:
        img_array = np.frombuffer(img_bytes, np.uint8)
        img = cv2.imdecode(img_array, cv2.IMREAD_GRAYSCALE)

    return img


def preprocess_image(img):
    img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img = clahe.apply(img)

    img = cv2.resize(img, IMG_SIZE, interpolation=cv2.INTER_LINEAR)

    return img


# Test it
test_key = "raw-images/train/NORMAL/0004cfab-14fd-4e49-80ba-63a80b6bddd6.dcm"
test_img = load_image_from_s3(test_key)
processed = preprocess_image(test_img)
print(f"Raw: {test_img.shape} → Preprocessed: {processed.shape}")

Raw: (1024, 1024) → Preprocessed: (512, 512)


### 5.3 Read metadata back from Athena

In [9]:
#Get metadata from Athena

import awswrangler as wr

df_meta = wr.athena.read_sql_query(
    "SELECT * FROM pneumonia_db.image_metadata",
    database="pneumonia_db"
)

# Remove checkpoint files if any
df_meta = df_meta[~df_meta["s3_key"].str.contains(".ipynb_checkpoint")]

print(f"Total images to process: {len(df_meta)}")

2026-06-03 21:48:16,686	WARNING services.py:2137 -- WARNING: The object store is using /tmp instead of /dev/shm because /dev/shm has only 891269120 bytes available. This will harm performance! You may be able to free up space by deleting files in /dev/shm. If you are inside a Docker container, you can increase /dev/shm size by passing '--shm-size=1.95gb' to 'docker run' (or add it to the run_options list in a Ray cluster config). Make sure to set this to more than 30% of available RAM.


2026-06-03 21:48:16,865	INFO worker.py:2007 -- Started a local Ray instance.


/opt/conda/lib/python3.12/site-packages/ray/_private/worker.py:2046: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


Total images to process: 33757


### 5.4 (Owner only) Full-dataset preprocessing

This block runs the preprocessing pipeline over the **entire** metadata set and uploads the results. Gated behind `IS_DATA_OWNER` so a re-run by anyone else is a no-op.

The smaller sampled run in §5.7 is the path graders / teammates should actually execute.


In [1]:
IS_DATA_OWNER = False

In [19]:

if IS_DATA_OWNER:
    manifest_rows = []
    errors = 0
    n = len(df_meta)
    print(f"Preprocessing {n} images...")
    
    for i, (_, row) in enumerate(df_meta.iterrows()):
        try:
            # Load raw image
            img = load_image_from_s3(row["s3_key"])
    
            # Preprocess
            processed = preprocess_image(img)
    
            # Calculate pixel stats (for normalization during training)
            pixel_mean = float(np.mean(processed))
            pixel_std = float(np.std(processed))
    
            # Save preprocessed image to S3
            preprocessed_key = f"preprocessed-images/{row['split']}/{row['label']}/{row['image_id']}.png"
            _, buf = cv2.imencode(".png", processed)
            s3.put_object(
                Bucket=bucket,
                Key=preprocessed_key,
                Body=buf.tobytes()
            )
    
            # Build manifest row
            manifest_rows.append({
                "image_id": str(row["image_id"]),
                "raw_s3_key": str(row["s3_key"]),
                "preprocessed_s3_key": preprocessed_key,
                "label": str(row["label"]),
                "label_int": int(1 if row["label"] == "PNEUMONIA" else 0),
                "split": str(row["split"]),
                "source": str(row["source"]),
                "file_type": str(row["file_type"]),
                "pixel_mean": round(pixel_mean, 4),
                "pixel_std": round(pixel_std, 4),
                "img_height": 512,
                "img_width": 512,
                "event_time": datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
            })
    
        except Exception as e:
            print(e)
            errors += 1
            continue
    
        if (i + 1) % 100 == 0:
            print(f"  Processed {i + 1}/{n}...")
    
    df_manifest = pd.DataFrame(manifest_rows)
    print(f"\n✅ Done! Preprocessed {len(df_manifest)} images ({errors} errors)")
    

Preprocessing 33757 images...


/tmp/ipykernel_1304/2149072017.py:43: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "event_time": datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")


  Processed 100/33757...


  Processed 200/33757...


  Processed 300/33757...


...

  Processed 33400/33757...


  Processed 33500/33757...


  Processed 33600/33757...


  Processed 33700/33757...



✅ Done! Preprocessed 33757 images (0 errors)


In [20]:
df_manifest.head(3)

,image_id,raw_s3_key,preprocessed_s3_key,label,label_int,split,source,file_type,pixel_mean,pixel_std,img_height,img_width,event_time
0,0004cfab-14fd-4e49-80ba-63a80b6bddd6,raw-images/train/NORMAL/0004cfab-14fd-4e49-80b...,preprocessed-images/train/NORMAL/0004cfab-14fd...,NORMAL,0,train,rsna,dcm,129.9480,72.6177,512,512,2026-06-03T21:58:09Z
1,0022995a-45eb-4cfa-9a59-cd15f5196c64,raw-images/train/NORMAL/0022995a-45eb-4cfa-9a5...,preprocessed-images/train/NORMAL/0022995a-45eb...,NORMAL,0,train,rsna,dcm,122.4426,63.1398,512,512,2026-06-03T21:58:09Z
2,0025d2de-bd78-4d36-9f72-e15a5e22ca82,raw-images/train/NORMAL/0025d2de-bd78-4d36-9f7...,preprocessed-images/train/NORMAL/0025d2de-bd78...,NORMAL,0,train,rsna,dcm,149.2288,63.3756,512,512,2026-06-03T21:58:09Z


In [22]:
df_manifest.shape

(33757, 13)

### 5.5 (Owner only) Republish the manifest CSV

After full-dataset preprocessing the manifest schema changes (gains `preprocessed_s3_key`, `pixel_mean`, etc.). Push the new CSV to the same S3 location so the Athena external table picks it up.


In [23]:
if IS_DATA_OWNER:
    # Save locally then upload
    csv_path = "image_metadata.csv"
    df_manifest.to_csv(csv_path, index=False)
    # Upload to shared s3 bucket
    s3.upload_file(csv_path, bucket, f"pneumonia-project/metadata/{csv_path}")
    print(f"✅ Updated Metadata uploaded to s3://{bucket}/pneumonia-project/metadata/{csv_path}")

✅ Updated Metadata uploaded to s3://pneumonia-data-set-group-4/pneumonia-project/metadata/image_metadata.csv


### 5.6 Recreate the Athena table with the preprocessed-image schema

Drop the old (pre-preprocessing) table definition and re-register against the same S3 prefix with the new column set.


In [25]:
database_name = "pneumonia_db"

In [26]:
drop_statement = f"""
DROP TABLE IF EXISTS {database_name}.image_metadata
"""
pd.read_sql(drop_statement, conn)

/tmp/ipykernel_1304/2168075215.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(drop_statement, conn)


""


In [ ]:
# Recreate the Athena table with the preprocessed-image schema. Note that `split`
# is again absent — it's added later in Step 7 by writing to a separate CSV.
statement = f"""
CREATE EXTERNAL TABLE IF NOT EXISTS {database_name}.image_metadata (
    image_id              STRING,
    raw_s3_key            STRING,
    preprocessed_s3_key   STRING,
    label                 STRING,
    label_int             TINYINT,
    source                STRING,
    file_type             STRING,
    pixel_mean            DOUBLE,
    pixel_std             DOUBLE,
    img_height            INT,
    img_width             INT,
    event_time            STRING
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ','
LOCATION 's3://{bucket}/pneumonia-project/metadata/'
TBLPROPERTIES ('skip.header.line.count'='1')
"""
pd.read_sql(statement, conn)
print(f'Recreated table {database_name}.image_metadata')

### 5.7 Sampled preprocessing run

Runs the same preprocessing pipeline on a 1,000-image random sample so the rest of the notebook is reproducible end-to-end in a few minutes. Bump `SAMPLE_SIZE` to `len(df_meta)` for the full dataset.

In [ ]:
#Preprocess images and save to S3
# 500 is only for testing purpose
# Change to len(df_meta) for full dataset while ready for training

SAMPLE_SIZE = 1000 

sample = df_meta.sample(SAMPLE_SIZE, random_state=42)
manifest_rows = []
errors = 0

print(f"Preprocessing {SAMPLE_SIZE} images...")

for i, (_, row) in enumerate(sample.iterrows()):
    try:
        # Load raw image
        img = load_image_from_s3(row["s3_key"])

        # Preprocess
        processed = preprocess_image(img)

        # Calculate pixel stats (for normalization during training)
        pixel_mean = float(np.mean(processed))
        pixel_std = float(np.std(processed))

        # Save preprocessed image to S3
        preprocessed_key = f"preprocessed-images/{row['label']}/{row['image_id']}.png"
        _, buf = cv2.imencode(".png", processed)
        s3.put_object(
            Bucket=default_bucket,
            Key=preprocessed_key,
            Body=buf.tobytes()
        )

        # Build manifest row
        manifest_rows.append({
            "image_id": str(row["image_id"]),
            "raw_s3_key": str(row["s3_key"]),
            "preprocessed_s3_key": f"s3://{default_bucket}/{preprocessed_key}",
            "label": str(row["label"]),
            "label_int": int(1 if row["label"] == "PNEUMONIA" else 0),
            "source": str(row["source"]),
            "file_type": str(row["file_type"]),
            "pixel_mean": round(pixel_mean, 4),
            "pixel_std": round(pixel_std, 4),
            "img_height": 512,
            "img_width": 512,
            "event_time": datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
        })

    except Exception as e:
        print(e)
        errors += 1
        continue

    if (i + 1) % 100 == 0:
        print(f"  Processed {i + 1}/{SAMPLE_SIZE}...")

df_manifest = pd.DataFrame(manifest_rows)
print(f"\n✅ Done! Preprocessed {len(df_manifest)} images ({errors} errors)")
df_manifest.head()

### 5.8 Coerce dtypes for Feature Store

Feature Store is strict about column types — string vs object, int32 vs int64. Force everything to the canonical types before defining the feature group.

In [ ]:
 #Fix data types for Feature Store

df_manifest["image_id"] = df_manifest["image_id"].astype(str)
df_manifest["raw_s3_key"] = df_manifest["raw_s3_key"].astype(str)
df_manifest["preprocessed_s3_key"] = df_manifest["preprocessed_s3_key"].astype(str)
df_manifest["label"] = df_manifest["label"].astype(str)
df_manifest["source"] = df_manifest["source"].astype(str)
df_manifest["file_type"] = df_manifest["file_type"].astype(str)
df_manifest["event_time"] = df_manifest["event_time"].astype(str)

df_manifest["label_int"] = df_manifest["label_int"].astype(int)
df_manifest["img_height"] = df_manifest["img_height"].astype(int)
df_manifest["img_width"] = df_manifest["img_width"].astype(int)
df_manifest["pixel_mean"] = df_manifest["pixel_mean"].astype(float)
df_manifest["pixel_std"] = df_manifest["pixel_std"].astype(float)

print("✅ Data types ready")
print(df_manifest.dtypes)

### 5.9 Define the Feature Group

In [ ]:
#Create Feature Group

feature_group_name = "pneumonia-training-manifest"

feature_group = FeatureGroup(
    name=feature_group_name,
    sagemaker_session=sess
)

feature_group.load_feature_definitions(data_frame=df_manifest)

print(f"Feature group: {feature_group_name}")
print(f"Columns:")
for feat_def in feature_group.feature_definitions:
    print(f"  {feat_def.feature_name}: {feat_def.feature_type}")

### 5.10 Create the Feature Group

First run creates the group; subsequent runs short-circuit on `ResourceInUse` (`'Resource Already Exists'`) so this is idempotent.

In [ ]:
#Create Feature Group in Feature Store

try:
    feature_group.create(
        s3_uri=f"s3://{default_bucket}/feature-store/",
        record_identifier_name="image_id",
        event_time_feature_name="event_time",
        role_arn=role,
        enable_online_store=True
    )
    print("Creating feature group (this may take a minute)...")
    
    status = feature_group.describe().get("FeatureGroupStatus")
    while status == "Creating":
        print(f"  Status: {status}...")
        time.sleep(5)
        status = feature_group.describe().get("FeatureGroupStatus")
except Exception as e:
    print(f'error code: {e.response["Error"]["Code"]}')
    if e.response["Error"]["Code"] == "ResourceInUse" and "Resource Already Exists" in str(e):
        print("Group already exists. Continuing...")



print(f"✅ Feature group status: {status}")

### 5.11 Ingest manifest into Feature Store

In [ ]:
print(f"Ingesting {len(df_manifest)} records into Feature Store...")

feature_group.ingest(
    data_frame=df_manifest,
    max_workers=3,
    wait=True
)

print("✅ Feature ingestion complete!")

### 5.12 Verify Feature Store offline + online stores

The offline store is backed by S3 + Glue (queryable via Athena); the online store is a low-latency record lookup. We hit both to confirm the ingest landed.

In [ ]:
## Check the actual table name and database that Feature Store created 
import time

# Wait longer for offline store to sync
print("Waiting 120 seconds for offline store to sync...")
time.sleep(120)

# Check the actual table name in the Glue catalog
feature_store_query = feature_group.athena_query()

# Print the table name Feature Store is using
print(f"Table name: {feature_store_query.table_name}")
print(f"Database: {feature_store_query.database}")
print(f"Catalog: {feature_store_query.catalog}")

In [ ]:
# Query the Feature Store offline store to verify the ingested data
feature_store_query.run(
    query_string=f'SELECT label, COUNT(*) as count FROM "pneumonia_training_manifest_1780368800" GROUP BY label',
    output_location=f"s3://{default_bucket}/feature-store/query_results/"
)

feature_store_query.wait()
df_result = feature_store_query.as_dataframe()
print("Feature Store contents:")
print(df_result)

In [ ]:
# Test retrieving a single record from the Feature Store online store
featurestore_runtime = boto3.client("sagemaker-featurestore-runtime", region_name=region)

sample_id = df_manifest["image_id"].iloc[0]

response = featurestore_runtime.get_record(
    FeatureGroupName=feature_group_name,
    RecordIdentifierValueAsString=sample_id
)

print(f"Record for image: {sample_id}")
for feature in response["Record"]:
    print(f"  {feature['FeatureName']}: {feature['ValueAsString']}")

## Step 6 · Canonical Train / Val / Test / Production Split

Stratified 40 / 10 / 10 / 40 split on the preprocessed manifest. The production slice is reserved to simulate incoming inference traffic for the monitoring notebook (`Model_Monitoring.ipynb`) — it's *not* a held-out set for the CNN to peek at.

Stratification on `label` keeps the Normal:Pneumonia ratio consistent across splits, which matters because the dataset is ~2:1 imbalanced.


### 6.1 Build the three-way split

In [ ]:
# Split feature data into train(40%), test(10%), val(10%), production(40%)
from sklearn.model_selection import train_test_split

# Start with the full manifest
df = df_manifest.copy()

print(f"Total images: {len(df)}")
print(f"Label distribution:\n{df['label'].value_counts()}\n")

# First split: model data (60%) vs production (40%)
df_model, df_production = train_test_split(
    df, test_size=0.40, random_state=42, stratify=df["label"]
)

# Second split: train (40%) vs temp (20%) from the model data
# 40/60 = 0.667 of model data goes to train
df_train, df_temp = train_test_split(
    df_model, test_size=0.333, random_state=42, stratify=df_model["label"]
)

# Third split: test (10%) vs val (10%) from temp
df_test, df_val = train_test_split(
    df_temp, test_size=0.5, random_state=42, stratify=df_temp["label"]
)

# Assign new split labels
df_train["split"] = "train"
df_test["split"] = "test"
df_val["split"] = "val"
df_production["split"] = "production"

# Combine back
df_split = pd.concat([df_train, df_test, df_val, df_production], ignore_index=True)

print("Split results:")
print(f"  Train:      {len(df_train)} ({len(df_train)/len(df)*100:.1f}%)")
print(f"  Test:       {len(df_test)} ({len(df_test)/len(df)*100:.1f}%)")
print(f"  Val:        {len(df_val)} ({len(df_val)/len(df)*100:.1f}%)")
print(f"  Production: {len(df_production)} ({len(df_production)/len(df)*100:.1f}%)")
print(f"  Total:      {len(df_split)}")

### 6.2 Verify stratification held

In [ ]:
# Verify label ratio is consistent across all splits
print("Label distribution per split:\n")
for split_name in ["train", "test", "val", "production"]:
    subset = df_split[df_split["split"] == split_name]
    normal = len(subset[subset["label"] == "NORMAL"])
    pneumonia = len(subset[subset["label"] == "PNEUMONIA"])
    total = len(subset)
    print(f"  {split_name:12s} → Normal: {normal} ({normal/total*100:.1f}%)  Pneumonia: {pneumonia} ({pneumonia/total*100:.1f}%)")

### 6.3 Visualize the splits

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Split sizes
split_sizes = df_split["split"].value_counts()
axes[0].bar(split_sizes.index, split_sizes.values, color=["steelblue", "salmon", "mediumseagreen", "orange"])
axes[0].set_title("Number of Images per Split")
axes[0].set_ylabel("Count")
for i, v in enumerate(split_sizes.values):
    axes[0].text(i, v + 5, str(v), ha="center", fontweight="bold")

# Label distribution per split
split_label = df_split.groupby(["split", "label"]).size().unstack(fill_value=0)
split_label.plot(kind="bar", ax=axes[1], color=["steelblue", "salmon"])
axes[1].set_title("Label Distribution per Split")
axes[1].set_ylabel("Count")
axes[1].tick_params(axis="x", rotation=0)
axes[1].legend(title="Label")

plt.tight_layout()
plt.show()

## Step 7 · Benchmark Model in SageMaker

Floor for the CNN: a SKLearn logistic regression / random forest trained on **just** `pixel_mean` and `pixel_std`. If the CNN can't beat this comfortably, something's wrong with the CNN — these two summary statistics carry almost no diagnostic information about pneumonia.

Runs as a SageMaker SKLearn training job so we exercise the same training-job plumbing the CNN will use later.


### 7.1 SageMaker session + SKLearn estimator imports

In [ ]:
import os
import json
import boto3
import sagemaker
import pandas as pd
import numpy as np

from sagemaker.sklearn.estimator import SKLearn

sess = sagemaker.Session()
role = sagemaker.get_execution_role()
region = boto3.Session().region_name
s3 = boto3.client("s3")

print(f"Region: {region}")
print(f"Role: {role}")
print(f"Default bucket: {default_bucket}")
print(f"Data bucket: {bucket}")

### 7.2 Carve features from the manifest

In [ ]:
# Prepare simple benchmark training data

benchmark_features = ["pixel_mean", "pixel_std"]
target_col = "label_int"

required_cols = benchmark_features + [target_col, "split"]

missing_cols = [col for col in required_cols if col not in df_split.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df_benchmark = df_split[required_cols].copy()

print(df_benchmark.head())
print(df_benchmark["split"].value_counts())

### 7.3 Materialize train/val/test CSVs

SageMaker's SKLearn container expects CSVs with the target column included — produce one file per split.

In [ ]:
# Create train, validation, and test CSV files
# SageMaker SKLearn expects the target column to be included in the CSV

benchmark_dir = "benchmark-data"
os.makedirs(benchmark_dir, exist_ok=True)

train_df = df_benchmark[df_benchmark["split"] == "train"].drop(columns=["split"])
val_df = df_benchmark[df_benchmark["split"] == "val"].drop(columns=["split"])
test_df = df_benchmark[df_benchmark["split"] == "test"].drop(columns=["split"])

train_path = f"{benchmark_dir}/train.csv"
val_path = f"{benchmark_dir}/validation.csv"
test_path = f"{benchmark_dir}/test.csv"

train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path, index=False)
test_df.to_csv(test_path, index=False)

print(f"Train shape: {train_df.shape}")
print(f"Validation shape: {val_df.shape}")
print(f"Test shape: {test_df.shape}")

### 7.4 Upload training inputs to S3

In [ ]:
# Upload benchmark data to S3

benchmark_prefix = "pneumonia-project/benchmark-model"

train_s3_uri = sess.upload_data(
    path=train_path,
    bucket=default_bucket,
    key_prefix=f"{benchmark_prefix}/data/train"
)

val_s3_uri = sess.upload_data(
    path=val_path,
    bucket=default_bucket,
    key_prefix=f"{benchmark_prefix}/data/validation"
)

test_s3_uri = sess.upload_data(
    path=test_path,
    bucket=default_bucket,
    key_prefix=f"{benchmark_prefix}/data/test"
)

print("Training data:", train_s3_uri)
print("Validation data:", val_s3_uri)
print("Test data:", test_s3_uri)

### 7.5 Write the SageMaker training script

The estimator runs this script inside the SKLearn container. It loads the CSVs, fits the model, evaluates on val/test, and writes the artifact to `/opt/ml/model`.

In [ ]:
# Create SageMaker training script for benchmark model

script_dir = "benchmark_script"
os.makedirs(script_dir, exist_ok=True)

train_script = f"{script_dir}/train_benchmark.py"

script_content = '''
import argparse
import os
import json
import joblib
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix


def calculate_specificity(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp) if (tn + fp) > 0 else 0.0


if __name__ == "__main__":
    parser = argparse.ArgumentParser()

    parser.add_argument("--max-iter", type=int, default=1000)
    parser.add_argument("--random-state", type=int, default=42)

    args = parser.parse_args()

    train_dir = os.environ["SM_CHANNEL_TRAIN"]
    val_dir = os.environ["SM_CHANNEL_VALIDATION"]
    test_dir = os.environ["SM_CHANNEL_TEST"]
    model_dir = os.environ["SM_MODEL_DIR"]

    train_file = os.path.join(train_dir, "train.csv")
    val_file = os.path.join(val_dir, "validation.csv")
    test_file = os.path.join(test_dir, "test.csv")

    train_df = pd.read_csv(train_file)
    val_df = pd.read_csv(val_file)
    test_df = pd.read_csv(test_file)

    feature_cols = ["pixel_mean", "pixel_std"]
    target_col = "label_int"

    X_train = train_df[feature_cols]
    y_train = train_df[target_col]

    X_val = val_df[feature_cols]
    y_val = val_df[target_col]

    X_test = test_df[feature_cols]
    y_test = test_df[target_col]

    model = LogisticRegression(
        max_iter=args.max_iter,
        random_state=args.random_state,
        class_weight="balanced"
    )

    model.fit(X_train, y_train)

    val_preds = model.predict(X_val)
    test_preds = model.predict(X_test)

    val_metrics = {
        "validation_accuracy": accuracy_score(y_val, val_preds),
        "validation_precision": precision_score(y_val, val_preds, zero_division=0),
        "validation_recall": recall_score(y_val, val_preds, zero_division=0),
        "validation_f1": f1_score(y_val, val_preds, zero_division=0),
        "validation_specificity": calculate_specificity(y_val, val_preds)
    }

    test_metrics = {
        "test_accuracy": accuracy_score(y_test, test_preds),
        "test_precision": precision_score(y_test, test_preds, zero_division=0),
        "test_recall": recall_score(y_test, test_preds, zero_division=0),
        "test_f1": f1_score(y_test, test_preds, zero_division=0),
        "test_specificity": calculate_specificity(y_test, test_preds)
    }

    metrics = {**val_metrics, **test_metrics}

    print("Benchmark Model Metrics")
    print(json.dumps(metrics, indent=4))

    for key, value in metrics.items():
        print(f"{key}: {value}")

    joblib.dump(model, os.path.join(model_dir, "model.joblib"))
'''

with open(train_script, "w") as f:
    f.write(script_content)

print(f"Created training script: {train_script}")

### 7.6 Configure the SKLearn Estimator

In [ ]:
# Create SageMaker SKLearn Estimator

benchmark_output_path = f"s3://{default_bucket}/{benchmark_prefix}/output"

benchmark_estimator = SKLearn(
    entry_point="train_benchmark.py",
    source_dir=script_dir,
    role=role,
    instance_type="ml.m5.large",
    instance_count=1,
    framework_version="1.2-1",
    py_version="py3",
    output_path=benchmark_output_path,
    hyperparameters={
        "max-iter": 1000,
        "random-state": 42
    },
    metric_definitions=[
        {"Name": "validation_accuracy", "Regex": "validation_accuracy: ([0-9\\.]+)"},
        {"Name": "validation_precision", "Regex": "validation_precision: ([0-9\\.]+)"},
        {"Name": "validation_recall", "Regex": "validation_recall: ([0-9\\.]+)"},
        {"Name": "validation_f1", "Regex": "validation_f1: ([0-9\\.]+)"},
        {"Name": "validation_specificity", "Regex": "validation_specificity: ([0-9\\.]+)"},
        {"Name": "test_accuracy", "Regex": "test_accuracy: ([0-9\\.]+)"},
        {"Name": "test_precision", "Regex": "test_precision: ([0-9\\.]+)"},
        {"Name": "test_recall", "Regex": "test_recall: ([0-9\\.]+)"},
        {"Name": "test_f1", "Regex": "test_f1: ([0-9\\.]+)"},
        {"Name": "test_specificity", "Regex": "test_specificity: ([0-9\\.]+)"}
    ]
)

print("Benchmark estimator ready.")
print("Output path:", benchmark_output_path)

### 7.7 Launch the training job

Blocks until the job completes (a few minutes on `ml.m5.large`). The job logs stream into the notebook output.

In [ ]:
# Run SageMaker training job

benchmark_estimator.fit(
    {
        "train": train_s3_uri,
        "validation": val_s3_uri,
        "test": test_s3_uri
    }
)

### 7.8 Job summary

In [ ]:
# Show completed training job info

training_job_name = benchmark_estimator.latest_training_job.name

print("Benchmark training job completed.")
print("Training job name:", training_job_name)
print("Model artifact location:")
print(benchmark_estimator.model_data)

## Step 8 · Hand-off to the CNN Notebook

The CNN training lives in [`CNN_Model.ipynb`](CNN_Model.ipynb) — it picks up the preprocessed manifest from the Feature Store / Athena, builds `tf.data` pipelines per split, trains the 4-block CNN, and saves the best checkpoint to S3 for the monitoring notebook to deploy.
